# Finding Heavy Traffic Indicators on I-94

In this project, we analyze a dataset about the westbound traffic on the [I-94 Interstate highway](https://en.wikipedia.org/wiki/Interstate_94). The data was recorded hourly by a station located approximately midway between Minneapolis and Saint Paul, and it was made available by John Hogue on the [UCI Machine Learning Repository](https://archive.ics.uci.edu/ml/datasets/Metro+Interstate+Traffic+Volume).

The goal of our analysis is to determine a few indicators of heavy traffic on I-94. These indicators can be weather type, time of the day, time of the week, etc. Because the station only records westbound traffic near that location, our results should not be generalized to the entire I-94 highway.

## The I-94 Traffic Dataset

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

i_94 = pd.read_csv('Metro_Interstate_Traffic_Volume.csv')
i_94.head()

In [ ]:
i_94.tail()

In [ ]:
i_94.info()

The dataset has 48,204 rows and 9 columns. The only column with null values is `holiday` (only 61 non-null values) — a null there simply means the day wasn't a holiday. Each row describes traffic and weather data for a specific hour — the data spans from October 2012 to September 2018.

## Analyzing Traffic Volume

In [ ]:
i_94['traffic_volume'].plot.hist()
plt.title('Distribution of Traffic Volume')
plt.xlabel('Traffic Volume (cars per hour)')
plt.show()

In [ ]:
i_94['traffic_volume'].describe()

The hourly traffic volume varies from 0 to 7,280 cars, with an average of about 3,260 cars.

- About 25% of the time, only 1,193 cars or fewer passed the station each hour — this probably happens during the night, or when a road is under construction.
- About 25% of the time, the traffic volume was four times as much (4,933 cars or more).

The histogram has two peaks (one near zero and one around 4,500–5,000 cars), which suggests that daytime and nighttime might influence the traffic volume. This gives our analysis an interesting direction: comparing daytime data with nighttime data.

## Traffic Volume: Day vs. Night

We'll divide the dataset into two parts:

- Daytime data: hours from 7 a.m. to 7 p.m. (12 hours)
- Nighttime data: hours from 7 p.m. to 7 a.m. (12 hours)

While this is not a perfect criterion for distinguishing between nighttime and daytime, it's a good starting point.

In [ ]:
i_94['date_time'] = pd.to_datetime(i_94['date_time'])

day = i_94.copy()[(i_94['date_time'].dt.hour >= 7) & (i_94['date_time'].dt.hour < 19)]
print(day.shape)

night = i_94.copy()[(i_94['date_time'].dt.hour >= 19) | (i_94['date_time'].dt.hour < 7)]
print(night.shape)

The two parts don't have the same number of rows (23,877 vs. 24,327) because a few hours are missing from the data. For instance, on 2012-10-10 the measurement at 03:00 is followed directly by the one at 06:00 — the hours in between are missing.

In [ ]:
plt.figure(figsize=(11, 3.5))

plt.subplot(1, 2, 1)
plt.hist(day['traffic_volume'])
plt.xlim(-100, 7500)
plt.ylim(0, 8000)
plt.title('Traffic Volume: Day')
plt.ylabel('Frequency')
plt.xlabel('Traffic Volume')

plt.subplot(1, 2, 2)
plt.hist(night['traffic_volume'])
plt.xlim(-100, 7500)
plt.ylim(0, 8000)
plt.title('Traffic Volume: Night')
plt.ylabel('Frequency')
plt.xlabel('Traffic Volume')

plt.show()

In [ ]:
day['traffic_volume'].describe()

In [ ]:
night['traffic_volume'].describe()

The histogram of daytime traffic volume is left skewed: most of the values are high — 75% of the time there were 4,252 cars or more passing the station each hour (the 25th percentile is 4,252).

The histogram of nighttime traffic volume is right skewed: most of the values are low — 75% of the time the number of cars passing the station each hour was 2,819 or fewer.

Although there are still some measurements over 5,000 cars per hour, the traffic at night is generally light. Our goal is to find indicators of heavy traffic, so we'll only focus on the daytime data moving forward.

## Time Indicators

One of the possible indicators of heavy traffic is time. There might be more people on the road in a certain month, on a certain day, or at a certain time of the day. We'll look at a few line plots showing how the traffic volume changes according to:

- Month
- Day of the week
- Time of day

In [ ]:
day['month'] = day['date_time'].dt.month
by_month = day.groupby('month').mean(numeric_only=True)
by_month['traffic_volume'].plot.line()
plt.title('Average Daytime Traffic Volume by Month')
plt.xlabel('Month')
plt.ylabel('Traffic Volume')
plt.show()

The traffic looks less heavy during cold months (November–February) and more intense during warm months (March–October), with one interesting exception: July. Is there anything special about July? Is traffic significantly less heavy in July each year? Let's check how the traffic volume changed each year in July.

In [ ]:
day['year'] = day['date_time'].dt.year
only_july = day[day['month'] == 7]
only_july.groupby('year').mean(numeric_only=True)['traffic_volume'].plot.line()
plt.title('Average Daytime Traffic Volume in July, by Year')
plt.xlabel('Year')
plt.ylabel('Traffic Volume')
plt.show()

Typically, the traffic is pretty heavy in July, similar to the other warm months. The only exception is 2016, which had a high decrease in traffic volume. One possible reason for this is road construction or a temporary closure near the station — we'd need outside information to confirm it.

As a tentative conclusion, we can say that warm months generally show heavier traffic compared to cold months. In a warm month, you can expect for each hour of daytime a traffic volume close to 5,000 cars.

In [ ]:
day['dayofweek'] = day['date_time'].dt.dayofweek
by_dayofweek = day.groupby('dayofweek').mean(numeric_only=True)
by_dayofweek['traffic_volume'].plot.line()
plt.title('Average Daytime Traffic Volume by Day of the Week')
plt.xticks(ticks=range(7), labels=['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'])
plt.xlabel('Day of the Week')
plt.ylabel('Traffic Volume')
plt.show()

Traffic volume is significantly heavier on business days (Monday–Friday). Except for Monday, we only see values over 5,000 during business days. Traffic is lighter on weekends, with values below 4,000 cars.

Let's now see what values we have based on time of the day. The weekends would drag down the average values, so we'll look at business days and weekends separately.

In [ ]:
day['hour'] = day['date_time'].dt.hour
business_days = day.copy()[day['dayofweek'] <= 4]  # 4 == Friday
weekend = day.copy()[day['dayofweek'] >= 5]  # 5 == Saturday
by_hour_business = business_days.groupby('hour').mean(numeric_only=True)
by_hour_weekend = weekend.groupby('hour').mean(numeric_only=True)

print(by_hour_business['traffic_volume'])
print(by_hour_weekend['traffic_volume'])

In [ ]:
plt.figure(figsize=(11, 3.5))

plt.subplot(1, 2, 1)
by_hour_business['traffic_volume'].plot.line()
plt.xlim(6, 20)
plt.ylim(1500, 6500)
plt.title('Traffic Volume by Hour: Monday–Friday')
plt.xlabel('Hour')
plt.ylabel('Traffic Volume')

plt.subplot(1, 2, 2)
by_hour_weekend['traffic_volume'].plot.line()
plt.xlim(6, 20)
plt.ylim(1500, 6500)
plt.title('Traffic Volume by Hour: Weekend')
plt.xlabel('Hour')
plt.ylabel('Traffic Volume')

plt.show()

At each hour of the day, the traffic volume is generally higher during business days compared to the weekends. As somehow expected, the rush hours are around 7 and 16 — when most people travel from home to work and back. We see volumes of over 6,000 cars at rush hours.

To summarize, we found a few time-related indicators of heavy traffic:

- The traffic is usually heavier during warm months (March–October) compared to cold months (November–February).
- The traffic is usually heavier on business days compared to weekends.
- On business days, the rush hours are around 7 and 16.

## Weather Indicators

Another possible indicator of heavy traffic is weather. The dataset provides us with a few useful columns about weather: `temp`, `rain_1h`, `snow_1h`, `clouds_all`, `weather_main`, `weather_description`.

A few of these columns are numerical, so let's start by looking up their correlation values with `traffic_volume`.

In [ ]:
day[['temp', 'rain_1h', 'snow_1h', 'clouds_all', 'traffic_volume']].corr()['traffic_volume']

In [ ]:
day.plot.scatter(x='traffic_volume', y='temp')
plt.ylim(230, 320)  # a few temperatures of 0 K are clearly data errors
plt.title('Temperature vs. Traffic Volume')
plt.show()

Temperature shows the strongest correlation with traffic volume, but only about +0.13 — a weak correlation. The scatter plot confirms there's no clear pattern. The other numerical weather columns (`rain_1h`, `snow_1h`, `clouds_all`) don't show any strong correlation either.

So none of the numerical weather columns seem like reliable indicators for heavy traffic. Let's look at the categorical weather columns next.

## Weather Types

In [ ]:
by_weather_main = day.groupby('weather_main').mean(numeric_only=True)
by_weather_main['traffic_volume'].plot.barh()
plt.title('Average Traffic Volume by Weather Type')
plt.xlabel('Traffic Volume')
plt.show()

There's no weather type where traffic volume exceeds 5,000 cars. This makes finding a heavy traffic indicator more difficult. Let's also group by `weather_description`, which has a more granular weather classification.

In [ ]:
by_weather_description = day.groupby('weather_description').mean(numeric_only=True)
by_weather_description['traffic_volume'].plot.barh(figsize=(5, 10))
plt.axvline(5000, color='red', linestyle='--')
plt.title('Average Traffic Volume by Weather Description')
plt.xlabel('Traffic Volume')
plt.show()

In [ ]:
by_weather_description[by_weather_description['traffic_volume'] > 5000]['traffic_volume']

It looks like there are three weather types where traffic volume exceeds 5,000:

- Shower snow
- Light rain and snow
- Proximity thunderstorm with drizzle

It's not clear why these weather types have the highest average traffic values — this is bad weather, but not that bad. Perhaps more people take their cars out of the garage when the weather is bad instead of riding a bike or walking.

## Conclusion

In this project, we tried to find a few indicators of heavy traffic on the I-94 Interstate highway (westbound, between Minneapolis and Saint Paul). We managed to find two types of indicators:

**Time indicators**

- The traffic is usually heavier during warm months (March–October) compared to cold months (November–February).
- The traffic is usually heavier on business days compared to the weekends.
- On business days, the rush hours are around 7 and 16.

**Weather indicators**

- Shower snow
- Light rain and snow
- Proximity thunderstorm with drizzle

Possible next steps include using the nighttime data to look for heavy traffic indicators, and finding more time and weather indicators.